# 프로토콜 v1.1 — E2b: Basora 외가 공개한 예측값과 PI-BDP의 짝지은 비교 (탐색 분석, CPU)

Basora 외(RESS 2025)가 figshare에 공개한 **원 논문의 시험 예측값**(DOI 10.6084/m9.figshare.24080643, CC BY 4.0)을 그대로 쓰고, 같은 창에서 PI-BDP와 비교합니다. 새 학습은 없습니다(v1.1 1절 E2b).

- **자료:** figshare 묶음의 `results/ncmapss/cache/test/predictions.parquet`. DS01–DS05의 시험 창 144,015개, 방법 6개(HNN 10회, MCD·LRT·FO·RAD 각 5회, DE 5개). DE는 원 저자가 HNN 10회로 만든 것이 그대로 들어 있습니다.
- **창의 사이클:** E2c bayesrul 실행 노트북이 원 전처리 함수로 복원한 DS01–DS05 창 대응표(`windows_rep.csv.gz`)를 (서브셋, 엔진, 창 번호)로 붙이고, 레이블이 모두 같은지 확인합니다.
- **채점 규칙은 E2c bayesrul 채점과 같습니다 (v1.1 4–6절, `e2c_실행기록_이탈.md` 13–16번).**
  - 건전 구간 상한 레이블과 엔진별 상한 [0, 상한]
  - PI-BDP는 t − 1 출력 − 1, 첫 사이클 창은 모든 모델에서 뺌
  - 점 예측은 실행 평균
  - 확률 지표는 창마다 실행별 점수의 평균
  - 계층 붓스트랩 100,000회, RMSE와 interval score
- **실행을 빼지 않습니다.** 원 논문은 HNN 10회 중 학습 곡선이 이상한 2회를 뺐다고 했으나 어느 것인지 밝히지 않았고, 공개 파일에는 10회가 모두 있습니다. 주 분석은 10회 모두를 씁니다. 시험 RMSE가 다른 실행보다 크게 높은 2회를 뺀 HNN은 **기술 통계로만** 함께 냅니다. 시험 결과로 고른 것이므로 검정하지 않습니다.
- **원 논문 표 7 확인:** 파일에서 방법별 RMSE·NLL(실행 평균 ± SD)을 다시 계산해 원 논문 표 7과 나란히 놓습니다(자료가 원 논문 결과인지 확인).

**실행 전:** figshare에서 받은 zip(약 407 MB)을 압축을 풀지 말고 드라이브 `PI-BDP_NCMAPSS/addendum_v11/e2b_figshare/`에 올리고, 이 노트북을 커밋한 주소를 `RUN_COMMIT_URL`에 넣으세요. 런타임은 CPU면 됩니다(고용량 RAM 권장).

In [ ]:
import os, sys, json, glob, time, hashlib, re, zipfile
import numpy as np, pandas as pd
from scipy.stats import norm
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'; TMP = '/content/e2b'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm'); TMP = os.environ.get('E2B_TMP', './e2b_tmp')
IN3 = f'{ROOT}/phase3'; BR = f'{ROOT}/addendum_v11/e2c_bayesrul'; FIG = f'{ROOT}/addendum_v11/e2b_figshare'; OUT = f'{ROOT}/addendum_v11'
os.makedirs(TMP, exist_ok=True)
RUN_COMMIT_URL = ''
assert re.match(r'https://github\.com/.+/commit/[0-9a-f]{7,40}$', RUN_COMMIT_URL), '이 노트북을 먼저 커밋하고 주소를 넣으세요.'
SUBSETS = ['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05']
QL = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975]; QC = [f'q{int(round(q * 1000)):03d}' for q in QL]
K = ['subset', 'unit', 'cycle']
METHODS = ['HNN', 'MCD', 'DE', 'LRT', 'FO', 'RAD']
PAPER7 = {'DE': (8.163, 0.136, 2.372, 0.017), 'MCD': (8.466, 0.216, 2.396, 0.024), 'HNN': (8.279, 0.181, 2.516, 0.057),
          'RAD': (8.619, 0.100, 3.181, 0.533), 'LRT': (8.551, 0.122, 4.553, 2.008), 'FO': (8.235, 0.113, 5.007, 0.465)}  # 원 논문 표 7: RMSE 평균, SD, NLL 평균, SD
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
REF3 = json.load(open(f'{IN3}/sha256_phase3.json'))
bad = [k for k, v in REF3.items() if os.path.exists(f'{IN3}/{k}') and sha(f'{IN3}/{k}') != v]; assert not bad, f'Phase 3 파일이 바뀜: {bad}'
REFR = json.load(open(f'{BR}/sha256_e2c_bayesrul.json'))
assert sha(f'{BR}/windows_rep.csv.gz') == REFR['files']['windows_rep.csv.gz'], '창 대응표가 실행 당시와 다름'
c90 = json.load(open(f'{ROOT}/phase1/calibration.json'))['c']['90']; EPS = 0.5
print('Phase 3 해시', len(REF3), '| 창 대응표 해시 확인 | c90', c90)

In [ ]:
# === figshare 자료 읽기 (zip 안의 시험 예측 파일만 꺼냄) ===
zips = sorted(glob.glob(f'{FIG}/*.zip')); assert len(zips) == 1, f'{FIG}에 zip 파일 하나를 올려 주세요: {zips}'
ZIP = zips[0]; ZIP_SHA = sha(ZIP)
with zipfile.ZipFile(ZIP) as z:
    mem = [n for n in z.namelist() if n.replace('\\', '/').endswith('cache/test/predictions.parquet')]
    assert len(mem) == 1, f'시험 예측 파일을 찾지 못함: {mem}'
    PQ = z.extract(mem[0], TMP)
PQ_SHA = sha(PQ)
F = pd.read_parquet(PQ, columns=['model', 'method', 'ds_id', 'unit_id', 'win_id', 'labels', 'preds', 'stds'])
n_dup = int(F.duplicated(['model', 'ds_id', 'unit_id', 'win_id']).sum())
F = F.drop_duplicates(['model', 'ds_id', 'unit_id', 'win_id'])        # 원 코드의 캐시 저장에서 생기는 모델 중복 (있으면 기록)
cnt = F.groupby('model').size()
print('zip', os.path.basename(ZIP), ZIP_SHA[:12], '| 시험 파일', mem[0], PQ_SHA[:12], '| 중복 행', n_dup)
print('모델별 창 수:', cnt.unique(), '| 방법별 실행 수:', F.groupby('method').model.nunique().to_dict())
EXPECT = int(os.environ.get('E2B_EXPECT_WINDOWS', 144015))   # 원 논문 표 3의 시험 창 수 (환경 변수는 로컬 시험용)
assert set(F.method.unique()) == set(METHODS) and (cnt == EXPECT).all(), '예상한 구성(방법 6개, 모델마다 144,015 창)이 아님'

In [ ]:
# === 원 논문 표 7과 비교 (자료 확인, 전체 시험 창, 원 코드의 NLL 정의: torch gaussian_nll_loss, eps 1e-6) ===
def nll(mu, sd, y):
    v = np.maximum(sd.astype('float64') ** 2, 1e-6); return float(np.mean(0.5 * (np.log(v) + (y - mu) ** 2 / v)))
PM = pd.DataFrame([dict(model=m, method=g.method.iloc[0], rmse=float(np.sqrt(np.mean((g.preds - g.labels) ** 2))),
                        nll=nll(g.preds.values, g.stds.values, g.labels.values)) for m, g in F.groupby('model')])
T7 = PM.groupby('method').agg(rmse_mean=('rmse', 'mean'), rmse_sd=('rmse', 'std'), nll_mean=('nll', 'mean'), nll_sd=('nll', 'std'), n=('rmse', 'size')).reset_index()
T7 = T7.assign(paper_rmse=T7.method.map(lambda m: PAPER7[m][0]), paper_rmse_sd=T7.method.map(lambda m: PAPER7[m][1]),
               paper_nll=T7.method.map(lambda m: PAPER7[m][2]))
PM.to_csv(f'{OUT}/e2b_figshare_per_model.csv', index=False); T7.to_csv(f'{OUT}/e2b_table7_check.csv', index=False)
print(T7.round(3).to_string(index=False)); print(); print(PM[PM.method == 'HNN'].round(3).to_string(index=False))

In [ ]:
# === 창에 사이클 붙이기, 레이블 확인, 같은 창에 모든 모델 놓기 ===
W = pd.read_csv(f'{BR}/windows_rep.csv.gz'); W['subset'] = W.file.str.replace('N-CMAPSS_', '', regex=False)
V0 = pd.concat([pd.read_csv(f'{IN3}/test_PI-BDP_V0_{s}.csv.gz') for s in SUBSETS], ignore_index=True)
eol = (V0.cycle + V0.true).groupby([V0.subset, V0.unit]).max().rename('eol').reset_index()
W = W.merge(eol, on=['subset', 'unit'], how='left'); assert W.eol.notna().all(), '시험 엔진이 맞지 않음'
assert np.allclose(W.rul_raw, W.eol - W.cycle), '원 RUL이 EOL - cycle과 맞지 않음'
W['cap'] = W.groupby(['subset', 'unit']).label.transform('max')
assert np.allclose(W.label, np.minimum(W.rul_raw, W.cap)), '건전 구간 상한 레이블이 맞지 않음'
assert not W.duplicated(['ds_id', 'unit', 'win_id']).any()
ref = F[F.model == 'HNN_000'][['ds_id', 'unit_id', 'win_id', 'labels']].rename(columns={'unit_id': 'unit'})
chk = W.merge(ref, on=['ds_id', 'unit', 'win_id'], how='outer', indicator=True)
assert (chk._merge == 'both').all(), f'figshare 창과 복원한 창이 다름: {chk._merge.value_counts().to_dict()}'
assert np.allclose(chk.labels, chk.label), 'figshare 레이블과 복원한 레이블이 다름'
first = W.cycle == W.groupby(['subset', 'unit']).cycle.transform('min'); n_first = int(first.sum())
B = W[~first].merge(V0.assign(cycle=V0.cycle + 1)[K + QC], on=K, how='left'); assert B[QC].notna().all().all(), 't-1 PI-BDP 출력이 없는 창이 있음'
KEY = ['ds_id', 'unit', 'win_id']; B = B.sort_values(KEY).reset_index(drop=True)
y = B.label.values; cap = B.cap.values
clip = lambda v: np.clip(v, 0, cap[:, None] if np.ndim(v) == 2 else cap)
print('시험 창', len(W), '| figshare 창과 일치, 레이블 일치 | 채점 창', len(B), '| 뺀 첫 사이클 창', n_first)

In [ ]:
# === 모델별 창 단위 예측과 점수 (E2c bayesrul 채점과 같은 규칙) ===
def pinball(Q, y):
    d = y[:, None] - Q; tau = np.array(QL)[None, :]
    return np.mean(np.maximum(tau * d, (tau - 1) * d), axis=1)
def iscore(L, U, y): return (U - L) + 20 * (L - y) * (y < L) + 20 * (y - U) * (y > U)
def scores(L, U, Q): return dict(cover=((y >= L) & (y <= U)).astype(float), width=U - L, iscore=iscore(L, U, y), pinball=pinball(Q, y))
Qp = clip(B[QC].values - 1); m = B.q500.values - 1
Lc = np.maximum(m - c90 * np.maximum(m - (B.q050.values - 1), EPS), 0); Uc = m + c90 * np.maximum((B.q950.values - 1) - m, EPS)
PT = {'PI-BDP V0': clip(m)}
PR = {'PI-BDP V0': scores(clip(Lc), clip(Uc), Qp), 'PI-BDP V0 (raw quantiles)': scores(Qp[:, 1], Qp[:, 7], Qp)}
z = norm.ppf(QL); PER_RUN = []
GROUPS = {meth: sorted(F[F.method == meth].model.unique()) for meth in METHODS}
GROUPS['HNN (8 runs, descriptive)'] = [mm for mm in GROUPS['HNN'] if mm not in
                                       PM[PM.method == 'HNN'].sort_values('rmse').model.iloc[8:].tolist()]   # 시험 RMSE가 가장 높은 2회 제외
for meth, models in GROUPS.items():
    mus, acc = [], None
    for mm in models:
        d = B[KEY].merge(F[F.model == mm].rename(columns={'unit_id': 'unit'})[KEY + ['preds', 'stds']], on=KEY, how='left')
        assert d.preds.notna().all(); mu, sd = d.preds.values.astype(float), d.stds.values.astype(float); mus.append(mu)
        Q = clip(mu[:, None] + sd[:, None] * z[None, :]); s = scores(Q[:, 1], Q[:, 7], Q)
        acc = s if acc is None else {kk: acc[kk] + s[kk] for kk in s}
        if '(' not in meth:
            PER_RUN.append(dict(method=meth, model=mm, rmse=float(np.sqrt(np.mean((clip(mu) - y) ** 2))), **{kk: float(v.mean()) for kk, v in s.items()}))
    PT[meth] = clip(np.mean(mus, axis=0)); PR[meth] = {kk: v / len(models) for kk, v in acc.items()}
PER_RUN = pd.DataFrame(PER_RUN)
print('HNN 기술 통계용 제외:', sorted(set(GROUPS['HNN']) - set(GROUPS['HNN (8 runs, descriptive)'])))

In [ ]:
# === 지표 표 ===
def rmse(e): return float(np.sqrt(np.mean(np.square(e))))
ORDER = ['PI-BDP V0'] + METHODS + ['HNN (8 runs, descriptive)']
rows = []
for name in ORDER:
    e = PT[name] - y; st = pd.Series(e).groupby(B.subset.values).apply(rmse)
    rows.append(dict(model=name, pooled_rmse=rmse(e), mean_subset_rmse=float(st.mean()), **{s: float(st[s]) for s in SUBSETS}))
POINT = pd.DataFrame(rows)
PROB = pd.DataFrame([dict(model=k, cov90=v['cover'].mean(), width90=v['width'].mean(), iscore90=v['iscore'].mean(), pinball9=v['pinball'].mean())
                     for k, v in PR.items()])
POINT.to_csv(f'{OUT}/e2b_point.csv', index=False); PROB.to_csv(f'{OUT}/e2b_prob.csv', index=False); PER_RUN.to_csv(f'{OUT}/e2b_per_run.csv', index=False)
print(POINT.round(3).to_string(index=False)); print(); print(PROB.round(3).to_string(index=False))

In [ ]:
# === 계층 붓스트랩 (서브셋 → 엔진), PI-BDP V0 - 각 방법 (주 분석: 실행을 빼지 않은 6개 방법) ===
def hier_boot(ea, eb, subset, unit, kind, n_boot=100000, seed=1):
    va, vb = (ea ** 2, eb ** 2) if kind == 'rmse' else (ea, eb)
    U = pd.DataFrame({'subset': subset, 'unit': unit, 'a': va, 'b': vb}).groupby(['subset', 'unit']).agg(
        sa=('a', 'sum'), sb=('b', 'sum'), n=('a', 'size')).reset_index()
    subs = sorted(U.subset.unique()); rng = np.random.default_rng(seed)
    per = [U[U.subset == s][['sa', 'sb', 'n']].values for s in subs]
    SA = np.zeros(n_boot); SB = np.zeros(n_boot); N = np.zeros(n_boot)
    pick = rng.integers(0, len(subs), (n_boot, len(subs)))
    for j in range(len(subs)):
        for si, arr in enumerate(per):
            r = np.flatnonzero(pick[:, j] == si)
            if not len(r): continue
            tot = arr[rng.integers(0, len(arr), (len(r), len(arr)))].sum(1)
            SA[r] += tot[:, 0]; SB[r] += tot[:, 1]; N[r] += tot[:, 2]
    f = (lambda a, n: np.sqrt(a / n)) if kind == 'rmse' else (lambda a, n: a / n)
    d = f(SA, N) - f(SB, N); obs = float(f(U.sa.sum(), U.n.sum()) - f(U.sb.sum(), U.n.sum()))
    return dict(diff=obs, ci_lo=float(np.percentile(d, 2.5)), ci_hi=float(np.percentile(d, 97.5)),
                p_boot=float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))))
def wording(r):
    if r['ci_lo'] > 0: return 'PI-BDP less accurate'
    if r['ci_hi'] < 0: return 'PI-BDP more accurate (exploratory)'
    return 'comparable'
tests = []
for mdl in METHODS:
    r = hier_boot(PT['PI-BDP V0'] - y, PT[mdl] - y, B.subset.values, B.unit.values, 'rmse')
    tests.append(dict(metric='pooled RMSE', comparison=f'V0 - {mdl}', **r, wording=wording(r)))
    r = hier_boot(PR['PI-BDP V0']['iscore'], PR[mdl]['iscore'], B.subset.values, B.unit.values, 'mean')
    tests.append(dict(metric='interval score (90%)', comparison=f'V0 (calibrated) - {mdl}', **r, wording=wording(r)))
TESTS = pd.DataFrame(tests); TESTS.to_csv(f'{OUT}/e2b_tests.csv', index=False)
print(TESTS.round(4).to_string(index=False))

In [ ]:
# === 보조: 사이클 단위 상한 없는 RMSE (기술 통계) ===
cyc = V0[K + ['true', 'q500']].rename(columns={'q500': 'PI-BDP V0'})
for mdl in METHODS:
    g = F[F.method == mdl].groupby(['ds_id', 'unit_id', 'win_id'], as_index=False).preds.mean().rename(columns={'unit_id': 'unit'})
    cm = W[['ds_id', 'unit', 'win_id'] + K[0:1] + ['cycle']].merge(g, on=['ds_id', 'unit', 'win_id']).groupby(K, as_index=False).preds.mean().rename(columns={'preds': mdl})
    cyc = cyc.merge(cm, on=K, how='inner')
SEC = pd.DataFrame([dict(model=mm, cycle_rmse_uncapped=rmse(cyc[mm] - cyc.true), n_cycles=len(cyc)) for mm in ['PI-BDP V0'] + METHODS])
SEC.to_csv(f'{OUT}/e2b_cycle_level.csv', index=False); print(SEC.round(3).to_string(index=False))

In [ ]:
# === 요약 + 해시 ===
txt = '\n'.join([f"Addendum v1.1 E2b figshare comparison (exploratory) {time.strftime('%Y-%m-%d %H:%M:%S')}  run={RUN_COMMIT_URL}",
    f"Data: Basora et al., BayesRUL benchmark results, figshare doi:10.6084/m9.figshare.24080643 (CC BY 4.0); zip sha256 {ZIP_SHA}; {mem[0]} sha256 {PQ_SHA}; duplicate rows dropped {n_dup}",
    f"DS01-DS05 test windows {len(W)}; scored {len(B)}; first-cycle windows excluded {n_first}",
    '', 'Check against paper Table 7 (all test windows, uncapped, as published):', T7.round(3).to_string(index=False),
    '', 'Point predictions (healthy-phase capped labels, 0.1 Hz windows):', POINT.round(3).to_string(index=False),
    '', f'Probabilistic (PI-BDP calibrated with c90={c90}; multi-run methods averaged per window over runs):', PROB.round(3).to_string(index=False),
    '', 'Paired hierarchical bootstrap V0 - method (all published runs), 100,000 resamples:', TESTS.round(4).to_string(index=False),
    '', 'Secondary, cycle-level uncapped (descriptive):', SEC.round(3).to_string(index=False)])
open(f'{OUT}/e2b_summary.txt', 'w').write(txt)
H = {os.path.basename(p): sha(p) for p in sorted(glob.glob(f'{OUT}/e2b_*.*'))}
H['figshare_zip'] = ZIP_SHA; H['figshare_test_predictions'] = PQ_SHA
json.dump(H, open(f'{OUT}/sha256_e2b.json', 'w'), indent=1); print(txt)